# Team Season - team_dash_pt_shots

This notebook inspects the data **downloaded** from the `team_dash_pt_shots` endpoint.

Source folder: `02_data/01_raw/2025_26/02_team_season/team_dash_pt_shots`

Scope:
- Read parquet files in the folder
- Show how many files were downloaded and how many rows/columns they contain
- Report missing values (null cells) overall and by row/column distribution

Notes:
- Read-only analysis: no exports or writes


In [9]:
from pathlib import Path
import pandas as pd
import numpy as np

# Resolve project root by walking up to find 02_data
project_root = Path.cwd()
for _ in range(6):
    if (project_root / "02_data").exists():
        break
    project_root = project_root.parent

endpoint = "team_dash_pt_shots"
data_dir = project_root / "02_data" / "01_raw" / "2025_26" / "02_team_season" / endpoint

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)


In [10]:
files = sorted(data_dir.glob("*.parquet"))
print(f"Endpoint: {endpoint}")
print(f"Folder: {data_dir}")
print(f"Parquet files: {len(files)}")

files_df = pd.DataFrame({
    "file": [f.name for f in files],
    "size_mb": [round(f.stat().st_size / 1e6, 3) for f in files],
})
files_df


Endpoint: team_dash_pt_shots
Folder: /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/01_raw/2025_26/02_team_season/team_dash_pt_shots
Parquet files: 30


,file,size_mb
0,team_dash_pt_shots__team_id=1610612737.parquet,0.018
1,team_dash_pt_shots__team_id=1610612738.parquet,0.018
2,team_dash_pt_shots__team_id=1610612739.parquet,0.018
3,team_dash_pt_shots__team_id=1610612740.parquet,0.018
4,team_dash_pt_shots__team_id=1610612741.parquet,0.018
5,team_dash_pt_shots__team_id=1610612742.parquet,0.018
6,team_dash_pt_shots__team_id=1610612743.parquet,0.018
7,team_dash_pt_shots__team_id=1610612744.parquet,0.018
8,team_dash_pt_shots__team_id=1610612745.parquet,0.018
9,team_dash_pt_shots__team_id=1610612746.parquet,0.018


In [11]:
dfs = [pd.read_parquet(f) for f in files]

per_file = pd.DataFrame({
    "file": [f.name for f in files],
    "rows": [len(df) for df in dfs],
    "cols": [df.shape[1] for df in dfs],
})
per_file


,file,rows,cols
0,team_dash_pt_shots__team_id=1610612737.parquet,26,26
1,team_dash_pt_shots__team_id=1610612738.parquet,26,26
2,team_dash_pt_shots__team_id=1610612739.parquet,26,26
3,team_dash_pt_shots__team_id=1610612740.parquet,26,26
4,team_dash_pt_shots__team_id=1610612741.parquet,26,26
5,team_dash_pt_shots__team_id=1610612742.parquet,26,26
6,team_dash_pt_shots__team_id=1610612743.parquet,26,26
7,team_dash_pt_shots__team_id=1610612744.parquet,26,26
8,team_dash_pt_shots__team_id=1610612745.parquet,26,26
9,team_dash_pt_shots__team_id=1610612746.parquet,26,26


In [12]:
df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

print(f"Combined shape: {df.shape}")
print(f"Total rows: {len(df)}")
print(f"Total columns: {df.shape[1] if len(df.columns) else 0}")


Combined shape: (780, 26)
Total rows: 780
Total columns: 26


In [13]:
rows, cols = df.shape
total_cells = rows * cols
null_cells = int(df.isna().sum().sum()) if total_cells else 0
null_pct = (null_cells / total_cells) if total_cells else 0

summary = pd.DataFrame({
    "rows": [rows],
    "cols": [cols],
    "total_cells": [total_cells],
    "null_cells": [null_cells],
    "null_pct": [round(null_pct * 100, 3)],
})
summary


,rows,cols,total_cells,null_cells,null_pct
0,780,26,20280,3150,15.533


In [14]:
if df.empty:
    col_summary = pd.DataFrame()
else:
    col_nulls = df.isna().sum()
    col_null_pct = df.isna().mean()
    col_summary = (
        pd.DataFrame({
            "null_cells": col_nulls,
            "null_pct": (col_null_pct * 100).round(3),
        })
        .sort_values("null_pct", ascending=False)
    )

col_summary


,null_cells,null_pct
TOUCH_TIME_RANGE,690,88.462
SHOT_TYPE,660,84.615
DRIBBLE_RANGE,630,80.769
SHOT_CLOCK_RANGE,600,76.923
CLOSE_DEF_DIST_RANGE,540,69.231
FG3_PCT,30,3.846
TEAM_ID,0,0.000
FG3A_FREQUENCY,0,0.000
SEASON_TYPE,0,0.000
SEASON,0,0.000


In [15]:
if df.empty:
    row_dist = pd.DataFrame()
else:
    row_null_pct = df.isna().mean(axis=1)
    bins = [0, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 1.0]
    row_bins = pd.cut(row_null_pct, bins=bins, include_lowest=True)
    row_counts = row_bins.value_counts().sort_index()
    row_dist = pd.DataFrame({
        "rows": row_counts,
        "pct_rows": (row_counts / len(row_null_pct) * 100).round(3),
    })

row_dist


,rows,pct_rows
"(-0.001, 0.01]",0,0.0
"(0.01, 0.05]",0,0.0
"(0.05, 0.1]",0,0.0
"(0.1, 0.25]",780,100.0
"(0.25, 0.5]",0,0.0
"(0.5, 0.75]",0,0.0
"(0.75, 1.0]",0,0.0


In [16]:
# Merge all teams, split by TABLE_NAME, drop fully-null columns, report nulls, and export

files = sorted(data_dir.glob("*.parquet"))
dfs = [pd.read_parquet(f) for f in files]
df_all = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

if df_all.empty:
    print("No data to merge.")
else:
    table_col = "TABLE_NAME" if "TABLE_NAME" in df_all.columns else None
    if table_col is None:
        print("Required column not found: TABLE_NAME")
        print(df_all.columns)
    else:
        out_dir = project_root / "02_data" / "02_cleaned" / "2025_26" / "02_team_season"
        out_dir.mkdir(parents=True, exist_ok=True)

        table_values = sorted([v for v in df_all[table_col].dropna().unique()])

        null_summary = []
        for name in table_values:
            df_part = df_all.loc[df_all[table_col] == name].copy()

            if df_part.empty:
                print(f"\n{name}: no rows")
                fully_null = []
            else:
                col_nulls = df_part.isna().sum()
                col_null_pct = (df_part.isna().mean() * 100).round(3)
                cols_with_nulls = col_nulls[col_nulls > 0].index.tolist()
                fully_null = col_nulls[col_nulls == len(df_part)].index.tolist()

                print(f"\n{name}: columns with any nulls ({len(cols_with_nulls)})")
                if cols_with_nulls:
                    display(pd.DataFrame({
                        "null_cells": col_nulls[cols_with_nulls],
                        "null_pct": col_null_pct[cols_with_nulls],
                    }).sort_values("null_pct", ascending=False))

                print(f"{name}: fully null columns ({len(fully_null)})")
                if fully_null:
                    print(fully_null)

            # Drop fully-null columns (they belong to other tables)
            if fully_null:
                df_part = df_part.drop(columns=fully_null)

            # Recompute null summary AFTER dropping fully-null columns
            rows, cols = df_part.shape
            total_cells = rows * cols
            null_cells = int(df_part.isna().sum().sum()) if total_cells else 0
            null_pct = (null_cells / total_cells * 100) if total_cells else 0

            null_summary.append({
                "table_name": name,
                "rows": rows,
                "cols": cols,
                "total_cells": total_cells,
                "null_cells": null_cells,
                "null_pct": round(null_pct, 3),
            })

            out_path = out_dir / f"team_dash_pt_shots__{name}.parquet"
            df_part.to_parquet(out_path, index=False)
            print(f"Saved {name}: {len(df_part)} rows -> {out_path}")

        null_summary_df = pd.DataFrame(null_summary).sort_values("table_name")
        print("\nNull summary by TABLE_NAME (after dropping fully-null columns):")
        print(null_summary_df)



ClosestDefender10ftPlusShooting: columns with any nulls (4)


,null_cells,null_pct
SHOT_TYPE,120,100.0
SHOT_CLOCK_RANGE,120,100.0
DRIBBLE_RANGE,120,100.0
TOUCH_TIME_RANGE,120,100.0


ClosestDefender10ftPlusShooting: fully null columns (4)
['SHOT_TYPE', 'SHOT_CLOCK_RANGE', 'DRIBBLE_RANGE', 'TOUCH_TIME_RANGE']
Saved ClosestDefender10ftPlusShooting: 120 rows -> /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/02_cleaned/2025_26/02_team_season/team_dash_pt_shots__ClosestDefender10ftPlusShooting.parquet

ClosestDefenderShooting: columns with any nulls (4)


,null_cells,null_pct
SHOT_TYPE,120,100.0
SHOT_CLOCK_RANGE,120,100.0
DRIBBLE_RANGE,120,100.0
TOUCH_TIME_RANGE,120,100.0


ClosestDefenderShooting: fully null columns (4)
['SHOT_TYPE', 'SHOT_CLOCK_RANGE', 'DRIBBLE_RANGE', 'TOUCH_TIME_RANGE']
Saved ClosestDefenderShooting: 120 rows -> /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/02_cleaned/2025_26/02_team_season/team_dash_pt_shots__ClosestDefenderShooting.parquet

DribbleShooting: columns with any nulls (4)


...

GeneralShooting: fully null columns (4)
['SHOT_CLOCK_RANGE', 'DRIBBLE_RANGE', 'CLOSE_DEF_DIST_RANGE', 'TOUCH_TIME_RANGE']
Saved GeneralShooting: 120 rows -> /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/02_cleaned/2025_26/02_team_season/team_dash_pt_shots__GeneralShooting.parquet

ShotClockShooting: columns with any nulls (4)


,null_cells,null_pct
SHOT_TYPE,180,100.0
DRIBBLE_RANGE,180,100.0
CLOSE_DEF_DIST_RANGE,180,100.0
TOUCH_TIME_RANGE,180,100.0


ShotClockShooting: fully null columns (4)
['SHOT_TYPE', 'DRIBBLE_RANGE', 'CLOSE_DEF_DIST_RANGE', 'TOUCH_TIME_RANGE']
Saved ShotClockShooting: 180 rows -> /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/02_cleaned/2025_26/02_team_season/team_dash_pt_shots__ShotClockShooting.parquet

TouchTimeShooting: columns with any nulls (4)


,null_cells,null_pct
SHOT_TYPE,90,100.0
SHOT_CLOCK_RANGE,90,100.0
DRIBBLE_RANGE,90,100.0
CLOSE_DEF_DIST_RANGE,90,100.0


TouchTimeShooting: fully null columns (4)
['SHOT_TYPE', 'SHOT_CLOCK_RANGE', 'DRIBBLE_RANGE', 'CLOSE_DEF_DIST_RANGE']
Saved TouchTimeShooting: 90 rows -> /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/02_cleaned/2025_26/02_team_season/team_dash_pt_shots__TouchTimeShooting.parquet

Null summary by TABLE_NAME (after dropping fully-null columns):
                        table_name  rows  cols  total_cells  null_cells  \
0  ClosestDefender10ftPlusShooting   120    22         2640           0   
1          ClosestDefenderShooting   120    22         2640           0   
2                  DribbleShooting   150    22         3300           0   
3                  GeneralShooting   120    22         2640          30   
4                ShotClockShooting   180    22         3960           0   
5                TouchTimeShooting    90    22         1980           0   

   null_pct  
0     0.000  
1     0.000  
2     0.000  
3     1.136  
4     0.000  
5     0.000  
